In [2]:
# Filtering results based on average surfdiff score
import pandas as pd 
import os
import glob
import sys
import numpy as np

# Custom imports from utils.py
script_dir = os.path.dirname(os.getcwd())
repo_path = script_dir.split("WP2_Sweet_Protein_Discovery")[0] + "/WP2_Sweet_Protein_Discovery"
sys.path.append(repo_path)
from utils import find_disulfide_pairs, residues_to_check

rewrite = False

# Path to the SurfDiff results for all models
surfdiff_res = f"{repo_path}/6_structural_analyses/5_surfdiff_results"

# Read in the SurfDiff results for all models and extract the residue similarity scores
subdirs = os.listdir(surfdiff_res)
results = []
for sweet_protein in subdirs:
    res_path = os.path.join(surfdiff_res, sweet_protein, "scores")

    # List residue uniqueness score files and read in the residue similarity scores
    if not os.path.isdir(res_path):
        continue
    res_paths = glob.glob(res_path + "/*residue_uniqueness_scores.csv")
    print(sweet_protein, len(res_paths), "Files found")
    for res_file in res_paths:
        res_df = pd.read_csv(res_file, header = 0)
        model_name = os.path.basename(res_file)
        model_name = model_name.replace("_residue_uniqueness_scores.csv", "")    
        model_name = model_name.replace(sweet_protein + "_", "")
        model_name = model_name.lstrip("_")
        if "residue_similarity_score" in res_df.columns:
            results.append([sweet_protein, model_name, res_df["residue_similarity_score"].tolist()])

# Make results into a dataframe
surfdiff_results_df = pd.DataFrame(results, columns = ["query", "target_id", "similarity_scores"])

# Read in the representative sequence df from earlier results
repseq_df = pd.read_csv(os.path.join(repo_path, "6_structural_analyses", "4_openfold3_predictions", "surfdiff_input_structures.csv"))
repseq_df["target_id"] = [i if "|" not in i else i.split("|")[1] for i in repseq_df["target"]]

# Loop over the dataframe and calculate mean residue uniqueness
repseq_df = repseq_df.merge(surfdiff_results_df, on = ["query", "target_id"], how = "left")
repseq_df["avg_surfdiff_similarity"] = repseq_df["similarity_scores"].apply(np.mean)

# Calculate the average similarity score for the important residues for each model
for index, row in repseq_df.iterrows():
    important_residues = residues_to_check[row["query"]]
    important_residues = [i[1] for i in important_residues]
    if row["similarity_scores"] is np.nan:
        repseq_df.loc[index, "avg_sweet_res_similarity"] = np.nan
        continue
    important_res_scores = [row["similarity_scores"][i] for i in important_residues]
    repseq_df.loc[index, "avg_sweet_res_similarity"] = np.mean(important_res_scores)

repseq_df.sort_values(by=["query", "avg_sweet_res_similarity"], ascending = [True, False], inplace = True)
repseq_df

Brazzein_4HE7 18 Files found
Miraculin 588 Files found
Thaumatin-I_1LR2 810 Files found
Honey_truffle_sweetener 608 Files found
Thaumatin-II_3AOK 170 Files found
MNEI_1IV7 232 Files found


,query,target,of3_tm_score,of3_RMSD,of3_tm_align_start,of3_tm_align_end,of3_avg_pLDDT,of3_tm_align_seq_id,of3_tm_align_length,model,...,length_diff,interface_residues,grantham_score,pdb,cropped_tseq,best_model,target_id,similarity_scores,avg_surfdiff_similarity,avg_sweet_res_similarity
958,Brazzein_4HE7,UniProt_tr|A0ABQ7DUU8|A0ABQ7DUU8_BRACR,0.79879,1.83,0.0,58.0,76.450508,0.333,54,OF3,...,0.092593,"[('D', 1, 'S'), ('K', 4, 'L'), ('Y', 38, 'W'),...",70.250000,Brazzein_4HE7___A0ABQ7DUU8,QDSECLKEYGGDVGFGFCAPRIYPSFCVQRCRADKGAISGKCIWGQ...,Brazzein_4HE7___A0ABQ7DUU8_seed_42_sample_1_mo...,A0ABQ7DUU8,"[0.5910490420262771, 0.5535700986391776, 0.652...",0.585492,0.589359
168,Brazzein_4HE7,UniProt_tr|A0A8X8B8L9|A0A8X8B8L9_BRACI,0.77243,1.87,2.0,61.0,77.114839,0.278,54,AFDB,...,0.148148,"[('D', 1, 'Q'), ('K', 4, 'L'), ('Y', 38, 'T'),...",72.500000,Brazzein_4HE7___A0A8X8B8L9,AQDSECLKEYGGNVGFGFCAPQTFPTICYIRCREDKDAKGGRCITG...,Brazzein_4HE7___A0A8X8B8L9_seed_42_sample_5_mo...,A0A8X8B8L9,"[0.6383199799957717, 0.563512039521995, 0.6791...",0.516206,0.546190
165,Brazzein_4HE7,UniProt_tr|A0A8X7RL27|A0A8X7RL27_BRACI,0.77273,1.87,2.0,61.0,76.997258,0.278,54,AFDB,...,0.148148,"[('D', 1, 'Q'), ('K', 4, 'L'), ('Y', 38, 'T'),...",72.500000,Brazzein_4HE7___A0A8X7RL27,AQDSECLKEYGGNVGFGFCAPQTFPTICYIRCREDKDAKGGRCITG...,Brazzein_4HE7___A0A8X7RL27_seed_42_sample_5_mo...,A0A8X7RL27,"[0.6397156359076392, 0.5628317768170098, 0.678...",0.516618,0.544616
172,Brazzein_4HE7,UniProt_tr|A0A397Z8B5|A0A397Z8B5_BRACM,0.77727,1.80,2.0,60.0,76.684754,0.283,53,AFDB,...,0.129630,"[('D', 1, 'S'), ('K', 4, 'L'), ('Y', 38, 'T'),...",72.833333,Brazzein_4HE7___A0A397Z8B5,QDSECLKEYGGNVGFGFCAPRTFPTICYIRCREDKDAKGGRCITGD...,Brazzein_4HE7___A0A397Z8B5_seed_42_sample_2_mo...,A0A397Z8B5,"[0.3026833673365796, 0.1462449955317099, 0.534...",0.505734,0.494656
166,Brazzein_4HE7,UniProt_tr|A0A0D3BPK9|A0A0D3BPK9_BRAOL,0.74413,2.15,2.0,61.0,74.717097,0.302,53,AFDB,...,0.148148,"[('D', 1, 'Q'), ('K', 4, 'L'), ('Y', 38, 'T'),...",72.500000,Brazzein_4HE7___A0A0D3BPK9,AQDSECLKEYGGNVGFGFCAPRTFPTICYIRCREDKDAKGGRCITG...,Brazzein_4HE7___A0A0D3BPK9_seed_42_sample_5_mo...,A0A0D3BPK9,"[0.5722955394778331, 0.5359315719504254, 0.687...",0.512212,0.437204
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2361,Thaumatin-I_1LR2,OneKP_scaffold-XXYA-2021056-Verbascum_sp.,0.88862,1.76,0.0,218.0,90.105890,0.342,199,AF3,...,0.057971,"[('K', 66, 'H'), ('K', 77, 'G'), ('K', 96, 'F'...",92.000000,Thaumatin-I_1LR2___OneKP_scaffold-XXYA-2021056...,LILTVVNNCPFTIWPAIQPNAGHPTLERGGFPLTTLTHRSFPAPSA...,Thaumatin-I_1LR2___OneKP_scaffold-XXYA-2021056...,OneKP_scaffold-XXYA-2021056-Verbascum_sp.,"[0.3816888766368962, 0.3427540148358911, 1.0, ...",0.610521,0.404404
2084,Thaumatin-I_1LR2,OneKP_scaffold-GVCB-2008572-Oenothera_affinis-...,0.75263,2.13,0.0,194.0,80.418000,0.398,171,AF3,...,0.057971,"[('K', 66, 'Q'), ('K', 77, 'G'), ('K', 96, 'Q'...",77.125000,Thaumatin-I_1LR2___OneKP_scaffold-GVCB-2008572...,VTLTVRNNCPNTIWPATLTGAGQTQLPDTGFELVPGASAAITVPTH...,Thaumatin-I_1LR2___OneKP_scaffold-GVCB-2008572...,OneKP_scaffold-GVCB-2008572-Oenothera_affinis-...,"[0.0, 0.0, 1.0, 0.0345294078999511, 1.0, 0.045...",0.555131,0.403007
2316,Thaumatin-I_1LR2,OneKP_scaffold-VYGG-2008458-Stachyurus_praecox,0.90876,1.83,0.0,225.0,90.078451,0.338,204,OF3,...,0.091787,"[('K', 66, 'A'), ('K', 77, 'N'), ('K', 96, 'F'...",87.250000,Thaumatin-I_1LR2___OneKP_scaffold-VYGG-2008458...,FILTVVNNCHYTIWPAIQPNAGHPVLERGGFMLHALTHRSFPAPTQ...,Thaumatin-I_1LR2___OneKP_scaffold-VYGG-2008458...,OneKP_scaffold-VYGG-2008458-Stachyurus_praecox,"[0.1649530030486431, 0.2893666302141252, 1.0, ...",0.606457,0.396123
2060,Thaumatin-I_1LR2,OneKP_scaffold-FPCO-2005046-Interfilum_paradoxum,0.88183,2.07,0.0,225.0,92.628053,0.340,200,OF3,...,0.091787,"[('K', 66, 'E'), ('K', 77, 'E'), ('K', 96, 'L'...",72.250000,Thaumatin-I_1LR2___OneKP_scaffold-FPCO-2005046...,LTLHNQCQETIWPATFTTPGHNKVSGPPAPPLSPGDTFSYTVDAEW...,Thaumatin-I_1LR2___OneKP_scaffold

# Filter out sequences where all cysteines are not paired and remove sequences with uneven number of cysteines (except hits with only one cysteine)

In [ ]:
repseq_df["seq_length"] = repseq_df["cropped_tseq"].apply(lambda x: len(x.replace("-", "")))
repseq_df["cysteine_count"] = repseq_df["cropped_tseq"].apply(lambda x: x.count("C"))

for index, row in repseq_df.iterrows():
    pdb_file = f"{repo_path}/6_structural_analyses/4_openfold3_predictions/monomers/{row['query']}/{row['query']}___{row['target_id']}/" + f"{row['query']}___{row['target_id']}_seed_42_sample_1_model.pdb"
    pdb_file = pdb_file.replace("+", "_")
    disulfide_bridges = find_disulfide_pairs(pdb_file, chain = "A", use_actual_numbering = True)
    repseq_df.loc[index, "disulfide_bridges"] = str(disulfide_bridges.get("A", []))
    repseq_df.loc[index, "all_cysteines_paired"] = len(disulfide_bridges.get("A", [])) * 2 == row["cysteine_count"]

# Drop sequences with more than 1 cysteine and not all cysteines paired
targets_to_drop = []
for index, row in repseq_df.iterrows():
    if row["cysteine_count"] > 1 and not row["all_cysteines_paired"]:
        targets_to_drop.append(row["target_id"])

# Drop targets with unpaired cysteines
repseq_filtered_df = repseq_df[~repseq_df["target_id"].isin(targets_to_drop)]
print(repseq_filtered_df["query"].value_counts())

# For the second selection round, HTS hits were treated without the length difference threshold to allow for more candidates to be considered. 
# The rest of the candidates were filtered with a 10% length difference threshold. 
if rewrite:
    repseq_filtered_df = repseq_filtered_df[(repseq_filtered_df["length_diff"] <= 0.10) | (repseq_filtered_df["query"] == "Honey_truffle_sweetener")]
    repseq_test = repseq_filtered_df.copy()
    repseq_filtered_df.to_csv(os.path.join(repo_path, "8_protein_selection", "5_2nd_selection_round", "0_foldseek_candidates.tsv"), index = False, sep = "\t")

# Drop rows where length_diff is more than 10%
repseq_filtered_df = repseq_filtered_df[repseq_filtered_df["length_diff"] <= 0.10]
print(repseq_filtered_df["query"].value_counts())

# Keep only hits for Honey_truffle_sweetener and MNEI_1IV7
repseq_filtered_df = repseq_filtered_df[repseq_filtered_df["query"].isin(["Honey_truffle_sweetener", "MNEI_1IV7"])]

# Save tsv to 8_protein_selection
if rewrite:
    repseq_filtered_df.to_csv(os.path.join(repo_path, "8_protein_selection", "0_foldseek_candidates.tsv"), index = False, sep = "\t")

query
Thaumatin-I_1LR2           627
Miraculin                  433
Honey_truffle_sweetener    272
MNEI_1IV7                  222
Thaumatin-II_3AOK          133
Brazzein_4HE7               18
Name: count, dtype: int64
query
Thaumatin-I_1LR2           534
Miraculin                  403
MNEI_1IV7                  189
Thaumatin-II_3AOK          113
Honey_truffle_sweetener     20
Brazzein_4HE7                2
Name: count, dtype: int64
